# Einfuehrung: Wie denkt ein Sprachmodell?
In diesem Notebook erforschen wir, wie ein Sprachmodell (LLM) funktioniert. Wir nutzen dafuer Python und die Schnittstelle (**API**) von OpenAI.

Eine **API** (Application Programming Interface) ist wie ein Schalter: Wir schicken eine Anfrage ueber das Internet an den Server von OpenAI - und bekommen eine Antwort zurueck. Das passiert alles automatisch im Hintergrund.

**Wichtig:** Du musst nichts programmieren koennen. Achte nur auf die Texte in den Anfuehrungszeichen `"..."`.

In [ ]:
# Installation und Import

In [ ]:
pip install openai

In [ ]:
import openai
from openai import OpenAI

# HIER DEINEN API-KEY EINTRAGEN
client = OpenAI(api_key="")

---
## Teil 1: Der erste Kontakt
Wir senden eine einfache Anfrage an die KI. Das Modell verarbeitet den Text und gibt uns eine Antwort zurueck.

In [ ]:
frage = "Was ist der hoechste Berg in Bayern?"

antwort = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[{"role": "user", "content": frage}]
)

print("KI-Antwort: " + antwort.choices[0].message.content)

---
## Teil 2: Das Experiment - Gedaechtnis
Hat die KI ein Gedaechtnis? Wir verraten ihr unseren Namen und fragen im naechsten Schritt danach.

**Hypothese vor dem Ausfuehren:** Was wird die KI auf "Wie heisse ich?" antworten?  
Notiere deine Vermutung, bevor du die Zelle ausfuehrst.

In [ ]:
# Schritt A: Vorstellung
# Wir schicken unseren Namen - die Antwort der KI interessiert uns hier nicht,
# wir speichern sie daher absichtlich nicht.
prompt1 = "Hallo, mein Name ist Max Mustermann."
antwort1=client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[{"role": "user", "content": prompt1}]
)
print("Schritt A erledigt: Name wurde gesendet.")
print("KI-Antwort auf Schritt A: " + antwort1.choices[0].message.content)
print("=================================================")
# Schritt B: Nachfrage in einem voellig neuen, unabhaengigen Aufruf
prompt2 = "Wie heisse ich?"
antwort2 = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[{"role": "user", "content": prompt2}]
)

print("KI-Antwort auf Schritt B: " + antwort.choices[0].message.content)

**Was ist passiert?**  
Die KI hat den Namen nicht genannt - obwohl wir ihn gerade erst gesagt haben.  
Der Grund: Jeder API-Aufruf ist voellig unabhaengig. Das Modell hat **kein Gedaechtnis** zwischen zwei Anfragen.  
In der Fachsprache nennt man das: Das Modell ist **zustandslos** (*stateless*).

```
Aufruf 1:  ["Hallo, ich bin Max."]   --> KI antwortet, vergisst alles
Aufruf 2:  ["Wie heisse ich?"]        --> KI sieht nur diese eine Zeile
```

---
## Teil 3: Die Loesung - Simulation von Gedaechtnis
Um ein Gespraech zu fuehren, muessen wir der KI die gesamte bisherige Geschichte **jedes Mal neu mitschicken**.  
Die Variable `verlauf` enthaelt dabei drei moegliche Rollen:

| Rolle | Bedeutung |
|---|---|
| `system` | Gibt der KI eine Persoenlichkeit oder Verhaltensregeln |
| `user` | Nachrichten von uns |
| `assistant` | Vergangene Antworten der KI |

```
Aufruf 3:  [system, user: "Ich bin Max.", assistant: "Hallo Max!", user: "Wie heisse ich?"]  --> KI kennt den Verlauf
```

In [ ]:
verlauf = [
    {"role": "system",    "content": "Du bist ein hilfreicher Assistent."},
    {"role": "user",      "content": "Hallo, ich bin Max."},
    {"role": "assistant", "content": "Hallo Max! Schoen, dich kennenzulernen."},
    {"role": "user",      "content": "Welchen Namen habe ich dir gerade genannt?"}
]

antwort = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=verlauf
)

print("KI-Antwort mit Gedaechtnis: " + antwort.choices[0].message.content)

---
## Uebungsaufgaben

**Aufgabe 1:** Aendere die `system`-Rolle in einen muerrischen Piraten und fuehre die Zelle aus. Was aendert sich an Ton und Sprache der KI?

**Aufgabe 2:** Warum wird der Chat teurer, je laenger er dauert?  
*Tipp: Schau dir an, wie der `verlauf` aufgebaut ist - was muesste sich veraendern, wenn das Gespraech immer laenger wird?*

**Aufgabe 3:** Ergaenze den `verlauf` um ein Hobby und lass die KI danach fragen.

In [ ]:
# Aufgabe 1: Pirat
verlauf_pirat = [
    {"role": "system",    "content": "___"},  # <- Hier die Persona eintragen
    {"role": "user",      "content": "Hallo! Wer bist du?"},
]

antwort = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=verlauf_pirat
)

print("KI-Antwort: " + antwort.choices[0].message.content)

In [ ]:
# Aufgabe 3: Hobby im Verlauf ergaenzen
verlauf_hobby = [
    {"role": "system",    "content": "Du bist ein hilfreicher Assistent."},
    {"role": "user",      "content": "Hallo, ich bin Max."},
    {"role": "assistant", "content": "Hallo Max! Schoen, dich kennenzulernen."},
    {"role": "user",      "content": "___"},  # <- Hier das Hobby nennen
    {"role": "assistant", "content": "___"},  # <- Hier eine passende Antwort eintragen
    {"role": "user",      "content": "Was weisst du alles ueber mich?"}
]

antwort = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=verlauf_hobby
)

print("KI-Antwort: " + antwort.choices[0].message.content)